# Step 3 — Synthetic Messy-Resume Generation

Renders each held-out resume (from `heldout_ground_truth.csv`, produced by the DistilBERT notebook)
into **4 layout conditions** using `reportlab`, preserving identical text content while varying layout:

1. **Clean single-column** (control)
2. **Two-column layout**
3. **Table-based sections**
4. **Non-standard headers with icons/symbols**

Output: PDFs in `synthetic_resumes/` + a manifest CSV (`synthetic_resumes_manifest.csv`) mapping
each PDF to its resume_id, category, layout condition, and ground-truth text — this manifest is
what Step 4 (parsing/degradation measurement) will consume.


In [1]:
# Install dependency (skip if already installed)
import sys
!{sys.executable} -m pip install -q reportlab


In [2]:
import os
import re
import pandas as pd

from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from reportlab.platypus import (
    SimpleDocTemplate, BaseDocTemplate, PageTemplate, Frame,
    Paragraph, Spacer, Table, TableStyle, FrameBreak
)
from reportlab.lib import colors

HELDOUT_CSV = "heldout_ground_truth.csv"   # produced by the DistilBERT notebook (Step 1)
OUTPUT_DIR = "synthetic_resumes"
LIMIT = None   # set to a small number (e.g. 20) to test on a subset first; None = all held-out resumes

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Section-splitting helper

The dataset gives one text blob per resume (no pre-labeled sections), so we chunk each resume's
text into pseudo-sections purely for layout purposes — the goal is controlled layout variation
with identical content, not semantically accurate section labeling.

**Important fix:** chunks are capped at a max character length (`MAX_CHARS_PER_SECTION`), not a
fixed count of 4. This matters because some real resumes (especially after PDF text extraction)
have little or no sentence-ending punctuation — a naive split would treat the *entire resume* as
one giant "sentence" and dump it into a single table/frame cell, which overflows the page and
throws a `reportlab.platypus.doctemplate.LayoutError`. Capping by character length guarantees no
single chunk can ever be too large to fit, and text with no punctuation at all gets force-wrapped
by word count as a fallback. Tested against a resume with zero punctuation and ~10,800 characters
— it now spans multiple pages cleanly instead of crashing.

In [3]:
BASE_SECTION_LABELS = ["SUMMARY", "SKILLS", "EXPERIENCE", "EDUCATION"]
MAX_CHARS_PER_SECTION = 500   # hard cap so no single table/frame cell can ever overflow a page

styles = getSampleStyleSheet()
body_style = ParagraphStyle("Body", parent=styles["Normal"], fontSize=9.5, leading=13, alignment=TA_LEFT)
heading_style = ParagraphStyle("Heading", parent=styles["Heading3"], fontSize=11, spaceAfter=4)


def split_into_sections(text, max_chars=MAX_CHARS_PER_SECTION):
    """Split resume text into pseudo-sections, guaranteeing no chunk exceeds max_chars.
    Prevents LayoutErrors on resumes with little/no sentence-ending punctuation."""
    text = str(text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s for s in sentences if s]

    chunks = []
    current = ""
    for sent in sentences:
        if len(sent) > max_chars:
            # sentence itself too long (likely no punctuation) - force-wrap by words
            words = sent.split()
            piece = ""
            for w in words:
                if len(piece) + len(w) + 1 > max_chars:
                    if current:
                        chunks.append(current.strip())
                        current = ""
                    chunks.append(piece.strip())
                    piece = w
                else:
                    piece = f"{piece} {w}".strip()
            current = piece
            continue

        if len(current) + len(sent) + 1 > max_chars:
            chunks.append(current.strip())
            current = sent
        else:
            current = f"{current} {sent}".strip()

    if current:
        chunks.append(current.strip())

    if not chunks:
        chunks = [text[:max_chars]] if text else [""]

    return chunks


def label_chunks(chunks):
    """Assign labels to however many chunks resulted - cycles/extends beyond the
    base 4 labels for unusually long resumes instead of silently dropping content."""
    labels = []
    for i in range(len(chunks)):
        if i < len(BASE_SECTION_LABELS):
            labels.append(BASE_SECTION_LABELS[i])
        else:
            labels.append(f"ADDITIONAL {i - len(BASE_SECTION_LABELS) + 1}")
    return labels


## The 4 layout generators

In [4]:
# ---------- Condition 1: Clean single-column (control) ----------
def generate_clean(resume_id, category, sections, labels, out_path):
    doc = SimpleDocTemplate(out_path, pagesize=letter,
                             topMargin=0.6 * inch, bottomMargin=0.6 * inch,
                             leftMargin=0.7 * inch, rightMargin=0.7 * inch)
    story = [Paragraph(f"{category} — Resume {resume_id}", styles["Title"]), Spacer(1, 12)]
    for label, content in zip(labels, sections):
        story.append(Paragraph(label, heading_style))
        story.append(Paragraph(content, body_style))
        story.append(Spacer(1, 10))
    doc.build(story)


# ---------- Condition 2: Two-column layout ----------
def generate_two_column(resume_id, category, sections, labels, out_path):
    doc = BaseDocTemplate(out_path, pagesize=letter,
                           topMargin=0.6 * inch, bottomMargin=0.6 * inch,
                           leftMargin=0.5 * inch, rightMargin=0.5 * inch)
    page_w, page_h = letter
    col_w = (page_w - 1.2 * inch) / 2
    frame_left = Frame(0.5 * inch, 0.6 * inch, col_w, page_h - 1.4 * inch, id="left")
    frame_right = Frame(0.5 * inch + col_w + 0.2 * inch, 0.6 * inch, col_w, page_h - 1.4 * inch, id="right")
    doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame_left, frame_right])])

    story = [Paragraph(f"{category} — Resume {resume_id}", styles["Heading2"]), Spacer(1, 8)]
    for i, (label, content) in enumerate(zip(labels, sections)):
        story.append(Paragraph(label, heading_style))
        story.append(Paragraph(content, body_style))
        story.append(Spacer(1, 8))
        if i == len(sections) // 2 - 1:
            story.append(FrameBreak())
    doc.build(story)


# ---------- Condition 3: Table-based sections ----------
def generate_table_based(resume_id, category, sections, labels, out_path):
    doc = SimpleDocTemplate(out_path, pagesize=letter,
                             topMargin=0.6 * inch, bottomMargin=0.6 * inch,
                             leftMargin=0.6 * inch, rightMargin=0.6 * inch)
    story = [Paragraph(f"{category} — Resume {resume_id}", styles["Title"]), Spacer(1, 12)]
    rows = []
    for label, content in zip(labels, sections):
        rows.append([Paragraph(f"<b>{label}</b>", body_style), Paragraph(content, body_style)])
    table = Table(rows, colWidths=[1.3 * inch, 5.0 * inch])
    table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("BACKGROUND", (0, 0), (0, -1), colors.whitesmoke),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(table)
    doc.build(story)


# ---------- Condition 4: Non-standard headers with icons/symbols ----------
ICONS = ["\u2605", "\u2726", "\u25B6", "\u2727", "\u25C6"]  # star, sparkle, triangle, sparkle2, diamond

def generate_icon_headers(resume_id, category, sections, labels, out_path):
    doc = SimpleDocTemplate(out_path, pagesize=letter,
                             topMargin=0.6 * inch, bottomMargin=0.6 * inch,
                             leftMargin=0.7 * inch, rightMargin=0.7 * inch)
    story = [Paragraph(f"{ICONS[0]} {category} {ICONS[0]}", styles["Title"]), Spacer(1, 12)]
    for i, (label, content) in enumerate(zip(labels, sections)):
        icon = ICONS[i % len(ICONS)]
        story.append(Paragraph(f"{icon} {label} {icon}", heading_style))
        content_iconified = content.replace(". ", f" {ICONS[(i + 1) % len(ICONS)]} ")
        story.append(Paragraph(content_iconified, body_style))
        story.append(Spacer(1, 10))
    doc.build(story)


CONDITIONS = {
    "clean": generate_clean,
    "two_column": generate_two_column,
    "table_based": generate_table_based,
    "icon_headers": generate_icon_headers,
}


## Run generation across the held-out set

In [5]:
def generate_all(heldout_csv=HELDOUT_CSV, limit=LIMIT):
    df = pd.read_csv(heldout_csv)
    if "resume_id" not in df.columns:
        df.insert(0, "resume_id", range(len(df)))
    if limit:
        df = df.head(limit)

    manifest_rows = []
    for _, row in df.iterrows():
        resume_id = row["resume_id"]
        category = row["Category"]
        text = row["clean_text"] if "clean_text" in row and pd.notna(row["clean_text"]) else row["Resume"]
        sections = split_into_sections(text)
        labels = label_chunks(sections)

        for condition_name, gen_fn in CONDITIONS.items():
            out_path = os.path.join(OUTPUT_DIR, f"resume_{resume_id}_{condition_name}.pdf")
            gen_fn(resume_id, category, sections, labels, out_path)
            manifest_rows.append({
                "resume_id": resume_id,
                "category": category,
                "condition": condition_name,
                "pdf_path": out_path,
                "ground_truth_text": text,
            })

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv("synthetic_resumes_manifest.csv", index=False)
    print(f"Generated {len(manifest)} PDFs across {len(CONDITIONS)} conditions for {len(df)} resumes.")
    print("Manifest saved to synthetic_resumes_manifest.csv")
    return manifest

manifest = generate_all()
manifest.head(10)


Generated 580 PDFs across 4 conditions for 145 resumes.
Manifest saved to synthetic_resumes_manifest.csv


,resume_id,category,condition,pdf_path,ground_truth_text
0,0,Blockchain,clean,synthetic_resumes/resume_0_clean.pdf,Hobbies Playing Chess Solving Rubik s Cube Wat...
1,0,Blockchain,two_column,synthetic_resumes/resume_0_two_column.pdf,Hobbies Playing Chess Solving Rubik s Cube Wat...
2,0,Blockchain,table_based,synthetic_resumes/resume_0_table_based.pdf,Hobbies Playing Chess Solving Rubik s Cube Wat...
3,0,Blockchain,icon_headers,synthetic_resumes/resume_0_icon_headers.pdf,Hobbies Playing Chess Solving Rubik s Cube Wat...
4,1,Java Developer,clean,synthetic_resumes/resume_1_clean.pdf,"TECHNICAL SKILLS Skills: Java, SQL, PL/SQL, C,..."
5,1,Java Developer,two_column,synthetic_resumes/resume_1_two_column.pdf,"TECHNICAL SKILLS Skills: Java, SQL, PL/SQL, C,..."
6,1,Java Developer,table_based,synthetic_resumes/resume_1_table_based.pdf,"TECHNICAL SKILLS Skills: Java, SQL, PL/SQL, C,..."
7,1,Java Developer,icon_headers,synthetic_resumes/resume_1_icon_headers.pdf,"TECHNICAL SKILLS Skills: Java, SQL, PL/SQL, C,..."
8,2,Operations Manager,clean,synthetic_resumes/resume_2_clean.pdf,KEY COMPETENCIES Multi - Operations Management...
9,2,Operations Manager,two_column,synthetic_resumes/resume_2_two_column.pdf,KEY COMPETENCIES Multi - Operations Management...


---
### Next: Step 4 — Parsing and Degradation Measurement

Use `pdfplumber`/`PyMuPDF` to extract text back out of each PDF in `synthetic_resumes/`, then
compare against `ground_truth_text` in the manifest using edit-distance / word-overlap metrics
to compute a parsing degradation score per layout condition.
